In [3]:
from selenium import webdriver as wd
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException
import time
import pandas as pd 

# 브라우저 실행
driver = wd.Chrome()

try:
  # [Step 1] 스타벅스 매장 찾기 페이지 접속
  driver.get('https://www.starbucks.co.kr/store/store_map.do')
  time.sleep(10) # 초기 페이지 로딩 대기
  
  # [Step 2] '지역 검색' 탭 클릭
  driver.find_element(By.CSS_SELECTOR, '.loca_search').click()
  time.sleep(5)

  # [Step 3] 시/도 리스트 요소를 모두 가져오기
  sidos = driver.find_elements(By.CSS_SELECTOR, '.sido_arae_box > li > a')

  sido_list = [sido.text.strip() for sido in sidos]

  for sido in sido_list:
    # 리스트를 돌면서 텍스트가 '서울'인 요소를 찾음
    if sido == '서울':
      
      # 서울 클릭
      driver.find_element(By.CSS_SELECTOR, '.sido_arae_box > li > a').click()
      time.sleep(5)
      
      # 서울시 군/구 리스트에서 '전체' 클릭
      driver.find_element(By.CSS_SELECTOR, '.gugun_arae_box > li > a').click()
      time.sleep(10)
      
      # [Step 4] 매장 정보 추출 시작
      stores = driver.find_elements(By.CSS_SELECTOR, 'ul.quickSearchResultBoxSidoGugun li.quickResultLstCon')
      
      seoul_store_info_list = []
      
      for store in stores:
        # 매장 이름
        name = store.get_attribute('data-name')
        
        # 주소 및 전화번호 (p 태그 안에 주소와 전화번호가 같이 있음)
        address_raw = store.find_element(By.TAG_NAME, 'p').get_attribute('textContent')
        address = address_raw.replace('1522-3232', '').replace(',','|').strip() # 전화번호 제거 및 주소 안에 있는 쉼표 구분자 파이프로 치환
        
        # 매장 타입 (클래스명으로 구분)
        # i 태그의 클래스 속성에 'pin_general', 'pin_reserve' 등이 포함됨
        icon = store.find_element(By.TAG_NAME, 'i')
        icon_class = icon.get_attribute('class')
        
        store_type = icon_class.replace('pin_','').strip() # pin_ 문자 제거
       
        seoul_store_info_list.append([name, address, store_type])
        

      # [Step 5] 데이터 저장 (for 루프가 완전히 끝난 시점에서 수행)
      # 수집된 리스트가 비어있지 않을 때만 저장 진행
      if seoul_store_info_list:
          try:
            # 1) 리스트 데이터를 데이터프레임으로 전환 및 상단 제목 헤더 지정
            df = pd.DataFrame(seoul_store_info_list, columns=['매장명', '주소', '매장타입'])
                  
            # 2) 저장할 파일명 설정
            csv_file = "유성진.csv"
                  
            # 3) 데이터프레임을 CSV 파일로 내보내기
            df.to_csv(csv_file, index=False, encoding='utf-8-sig')
                  
            print(f"저장 완료: 총 {len(df)}개의 서울시 스타벅스 매장 정보를 수집했습니다.")
            
          except PermissionError: # 파일이 이미 열려 있어 수정이 불가능할 때 발생하는 예외입니다.
            print(f"파일 저장 실패: '{csv_file}' 파일이 이미 열려 있습니다. 파일을 닫고 다시 실행하세요.")
          
          except Exception as e: # 기타 Pandas 관련 예외 처리
                    
            print(f"데이터프레임 처리 중 예상치 못한 오류 발생: {e}")
              
          break # '서울'만 얻고 싶으므로 시/도 반복문 종료

except WebDriverException as e:
    # 브라우저가 강제로 닫히거나 인터넷 연결 문제 시 발생
    print(f"브라우저 제어 중 오류 발생: {e}")
    
except Exception as e:
    # 그 외 모든 예외 상황 처리
    print(f"프로그램 실행 중 시스템 오류 발생: {e}")
        
finally:
    # 브라우저 종료
    driver.quit()

저장 완료: 총 675개의 서울시 스타벅스 매장 정보를 수집했습니다.
